# 05 · Gate — 01 Grounding check

**Ported from `specialist-rag`'s `services/grounding.py` (305 L) and `tests/unit/test_grounding.py`, branch `origin/fix-completion` at commit `130ff868`, NOT on `main`. The cookbook had zero abstention or grounding code before this notebook -- a grep for abstain/abstention/insufficient-evidence returned nothing, while stage `06-bench` exists specifically to measure faithfulness. This is the missing solution to the problem that stage measures.**

Deterministic, not an LLM judge, for its first and most important pass:
fabricated quotes are caught by fuzzy-matching every quoted span in an
answer against the retrieved context; invented citations are caught by
checking every PMC URL and DOI in the answer against what was actually
retrieved. An optional LLM judge layers on top for claims that aren't
quoted verbatim, but the deterministic layer catches the two failure
modes that matter most without ever calling a model.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `norm_doi` | Normalizes a bare DOI or `doi.org` URL to one comparable form | `norm_doi("https://doi.org/10.1000/x\n")` → `"10.1000/x"` |
| `context_text` | Builds the evidence text a claim is checked against -- titles, journals and ids included, not just chunk bodies | `context_text(papers)` |
| `quote_supported` | Fuzzy-matches one quoted span against the retrieved context | `quote_supported(quote, context_norm)` |
| `deterministic_check` | Full offline pass: fabricated quotes + orphan PMC/DOI citations | `deterministic_check(answer_text, papers)` |
| `is_bibliographic` | The false-positive guard -- a paper's own title isn't a "fabricated claim" | `is_bibliographic("Pediatric burn management", papers)` |
| `check_grounding` | Orchestrates the deterministic pass with an optional LLM judge, handles abstention | `await check_grounding(query, answer, papers)` |
| `annotation_for` | The user-facing warning trailer when grounding is weak | `annotation_for(result)` |


In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(6):
    if (_p / "nbio.py").is_file():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")

import nbio

repo_root = nbio.bootstrap()
nbio.show_environment()

## Step 1 — the fixture: two retrieved papers, real shape

Ported verbatim from `test_grounding.py`'s own `PAPERS` fixture -- a
retrieved paper carries its chunk `text`, a citation (`pmc_url`, `doi`),
and bibliographic metadata (`title`, `journal`). Every test below checks
an answer against these same two papers.

In [ ]:
PAPERS = [
    {
        "text": "Early excision and grafting reduces mortality in major pediatric burns.",
        "pmc_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC123/",
        "doi": "10.1000/burns.123",
        "title": "Pediatric burn management",
        "journal": "Journal of Burn Care",
    },
    {
        "text": "Split-thickness skin grafts are preferred for large surface areas.",
        "pmc_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC456/",
        "doi": "10.1000/grafts.456",
        "title": "Grafting techniques",
        "journal": "Plastic and Reconstructive Surgery",
    },
]
print(f"{len(PAPERS)} retrieved papers, the fixture every check below runs against")

## Step 2 — normalization: strip escape artifacts, one canonical DOI form

Ported verbatim, including the reason each exists. `_strip_escapes` matters
because a model's answer can carry a literal `\n` (backslash-n as two
characters, not a real newline) that would otherwise break a citation
match. `norm_doi` matters because the same paper can be cited as a bare
DOI or as a `doi.org` URL, with or without an escape artifact tacked on --
all four forms must resolve to the same string, or a real citation gets
flagged as an orphan.

In [ ]:
import re

_ESCAPE_RE = re.compile(r"\\[nrt]")


def strip_escapes(s: str) -> str:
    """Replace literal escape sequences (\\n, \\t, \\r) with spaces and drop stray backslashes."""
    return _ESCAPE_RE.sub(" ", s or "").replace("\\", "")


def norm(s: str) -> str:
    return " ".join(strip_escapes(s).lower().split())


def norm_doi(s: str) -> str:
    """Normalize a DOI or doi.org URL to a bare lowercase DOI, free of escape/whitespace artifacts."""
    x = strip_escapes(s or "").lower()
    x = "".join(x.split())
    x = re.sub(r"^https?://(dx\.)?doi\.org/", "", x)
    return x.rstrip(".,;)]/").strip()


print("bare DOI:            ", norm_doi("10.1097/PRS.0b013"))
print("doi.org URL:         ", norm_doi("https://doi.org/10.1097/x.y"))
print("with escape artifact:", norm_doi("https://doi.org/10.1097/x.y\\n"))

assert norm_doi("https://doi.org/10.1097/x.y\\n") == "10.1097/x.y"
assert norm_doi("10.1097/PRS.0b013\\n") == "10.1097/prs.0b013"
print()
print("all three forms normalize to the same comparable string")

## Step 3 — `context_text`: what a claim is actually checked against

Ported verbatim. Concatenates each paper's bibliographic metadata (title,
journal, year, ids) alongside its chunk text -- so a correctly-cited
*reference* counts as grounded, not just a correctly-quoted *sentence*.
Deduplicates identical segments (a paper whose `text` and `source_text`
happen to be identical shouldn't count twice).

In [ ]:
def context_text(papers: list[dict]) -> str:
    blocks = []
    seen: set[str] = set()
    for p in papers:
        meta = " ".join(str(p.get(k) or "") for k in ("title", "journal", "year", "pmc_url", "doi"))
        for part in (meta, p.get("text") or "", p.get("source_text") or ""):
            block = norm(part)
            if block and block not in seen:
                blocks.append(block)
                seen.add(block)
    return "\n".join(blocks)


ctx = context_text(PAPERS)
print(ctx)

assert "pediatric burn management" in ctx
assert "plastic and reconstructive surgery" in ctx
print()
print("confirmed: titles and journals are part of the checkable context, not just chunk text")

## Step 4 — `quote_supported`: fuzzy-match one quoted span

Ported verbatim (`_QUOTE_MATCH_THRESHOLD = 0.85`). An exact substring
match is checked first; if that fails, `SequenceMatcher`'s longest
matching block must cover at least 85% of the quote's own length. Below
that, the quote is fabricated -- the model said something in quotation
marks that the retrieved evidence doesn't actually contain.

In [ ]:
from difflib import SequenceMatcher

QUOTE_MATCH_THRESHOLD = 0.85


def quote_supported(quote: str, context_norm: str) -> bool:
    q = norm(quote)
    if len(q) < 12:
        return True  # too short to judge; don't penalize
    if q in context_norm:
        return True
    sm = SequenceMatcher(None, q, context_norm)
    return sm.find_longest_match(0, len(q), 0, len(context_norm)).size >= int(len(q) * QUOTE_MATCH_THRESHOLD)


real_quote = "Split-thickness skin grafts are preferred for large surface areas."
fabricated_quote = "Hyperbaric oxygen cures all burns within three days guaranteed."

print("real quote supported?      ", quote_supported(real_quote, ctx))
print("fabricated quote supported?", quote_supported(fabricated_quote, ctx))

assert quote_supported(real_quote, ctx) is True
assert quote_supported(fabricated_quote, ctx) is False

## Step 5 — the two citation checks: orphan PMC URLs and orphan DOIs

Ported verbatim, including the regexes' own in-code comments about why
quotes/commas/backslashes are excluded from the match (a malformed
citation leaks JSON/escape junk right after the real URL, and that junk
must not become part of the reported orphan).

In [ ]:
_PMC_URL_RE = re.compile(r"https?://[^\s)\]\\\"\',]*pmc[^\s)\]\\\"\',]*", re.IGNORECASE)
_DOI_RE = re.compile(r"\b10\.\d{4,9}/[^\s)\]\"\'\\\\]+", re.IGNORECASE)


def orphan_pmc_urls(answer_text: str, papers: list[dict]) -> list[str]:
    retrieved = {norm(p.get("pmc_url") or "") for p in papers if p.get("pmc_url")}
    emitted = {norm(u).rstrip("\"\',.;)]") for u in _PMC_URL_RE.findall(answer_text or "")}
    return [u for u in sorted(emitted) if u and u not in retrieved]


def orphan_dois(answer_text: str, papers: list[dict]) -> list[str]:
    retrieved = {norm_doi(p.get("doi") or "") for p in papers if p.get("doi")}
    retrieved |= {norm_doi(p.get("pmc_url") or "") for p in papers if "doi.org" in (p.get("pmc_url") or "").lower()}
    retrieved.discard("")
    emitted = {norm_doi(u) for u in _DOI_RE.findall(answer_text or "")}
    return [f"https://doi.org/{d}" for d in sorted(emitted) if d and d not in retrieved]


retrieved_url_answer = "See https://pmc.ncbi.nlm.nih.gov/articles/PMC123/ for details."
orphan_url_answer = "See https://pmc.ncbi.nlm.nih.gov/articles/PMC999/ for details."
messy_orphan_answer = 'See https://pmc.ncbi.nlm.nih.gov/articles/PMC999/",\\"" for details.'

print("retrieved URL, flagged?", orphan_pmc_urls(retrieved_url_answer, PAPERS))
print("orphan URL, flagged?   ", orphan_pmc_urls(orphan_url_answer, PAPERS))
print("orphan + JSON junk:    ", orphan_pmc_urls(messy_orphan_answer, PAPERS))

assert orphan_pmc_urls(retrieved_url_answer, PAPERS) == []
assert orphan_pmc_urls(orphan_url_answer, PAPERS) == ["https://pmc.ncbi.nlm.nih.gov/articles/pmc999/"]
assert orphan_pmc_urls(messy_orphan_answer, PAPERS) == ["https://pmc.ncbi.nlm.nih.gov/articles/pmc999/"], (
    "the trailing JSON/escape junk must not leak into the flagged URL"
)
print()
print("all three citation checks correct, including the malformed-citation case")

## Step 6 — `deterministic_check`: both signals together

Ported verbatim. This is the whole offline pass: no model, no network,
runs on every answer regardless of whether a judge is configured.

In [ ]:
def deterministic_check(answer_text: str, papers: list[dict]) -> dict:
    context_norm = context_text(papers)
    quotes = re.findall(r'"([^"\n]{12,400})"', answer_text or "")
    unsupported = [q for q in quotes if not quote_supported(q, context_norm)]
    orphan = orphan_pmc_urls(answer_text, papers) + orphan_dois(answer_text, papers)
    return {"unsupported_quotes": unsupported[:5], "orphan_urls": orphan[:5], "quote_count": len(quotes)}


clean_answer = f'\U0001F4C4 "{real_quote}"'
fabricated_answer = f'\U0001F4C4 "{fabricated_quote}"'

print("clean answer:     ", deterministic_check(clean_answer, PAPERS))
print("fabricated answer:", deterministic_check(fabricated_answer, PAPERS))

assert deterministic_check(clean_answer, PAPERS)["unsupported_quotes"] == []
assert len(deterministic_check(fabricated_answer, PAPERS)["unsupported_quotes"]) == 1

## Step 7 — `is_bibliographic`: the hard half, false-positive care

Ported verbatim. Without this guard, a paper's own title or journal name
-- or a stray metadata token like `relevance_score` -- would be flagged
as a "fabricated clinical claim" the moment either the deterministic pass
or the LLM judge mentions it. This is what makes the difference between a
grounding check that's usable and one that cries wolf on its own
bibliography.

In [ ]:
_METADATA_TOKENS = {
    "relevance_score", "rcs_score", "citation_count", "citations",
    "journal_quality", "journalq", "evidence_level", "final_score", "pmc_url", "year", "score",
}


def token_key(s: str) -> str:
    return re.sub(r"[^a-z0-9_]", "", strip_escapes(s).lower())


def is_bibliographic(claim: str, papers: list[dict]) -> bool:
    if token_key(claim) in _METADATA_TOKENS:
        return True
    if claim and not re.search(r"[A-Za-z]", claim):
        return True
    c = norm(claim)
    if len(c) < 4:
        return False
    for p in papers:
        for field in ("title", "journal"):
            ref = norm(p.get(field) or "")
            if ref and (c == ref or SequenceMatcher(None, c, ref).ratio() >= 0.9):
                return True
    return False


genuine_claim = "DIEP flaps have a higher risk of fat necrosis than free TRAM flaps."
print("a paper title:      ", is_bibliographic("Pediatric burn management", PAPERS))
print("a metadata token:    ", is_bibliographic("relevance_score", PAPERS))
print("a genuine claim:     ", is_bibliographic(genuine_claim, PAPERS))
print("a stray score float: ", is_bibliographic(": 0.7240000000000001,", PAPERS))

assert is_bibliographic("Pediatric burn management", PAPERS)
assert is_bibliographic("relevance_score", PAPERS)
assert not is_bibliographic(genuine_claim, PAPERS)
assert is_bibliographic(": 0.7240000000000001,", PAPERS)

## Step 8 — abstention: `check_grounding` skips, on purpose, before anything else runs

Ported from `check_grounding`'s own first lines. Two cases short-circuit
the whole check, before the deterministic pass or any model call: the
answer is itself an abstention (`INSUFFICIENT EVIDENCE...`, this
notebook's counterpart to `04-benchmarks`'s `answered-without-evidence`
signal), or there were no retrieved papers to check against in the first
place. Checking this first, before anything else in the pipeline runs, is
the guardrail this whole notebook exists to prove works.

In [ ]:
_INSUFFICIENT_PREFIX = "INSUFFICIENT EVIDENCE"


def grounding_precondition(answer_text: str, papers: list[dict]) -> bool:
    """True if check_grounding should proceed at all."""
    text = (answer_text or "").strip()
    if not text or text.startswith(_INSUFFICIENT_PREFIX) or not papers:
        return False
    return True


print("abstained answer, papers present:", grounding_precondition("INSUFFICIENT EVIDENCE: none found", PAPERS))
print("real answer, no papers:          ", grounding_precondition("some grounded answer", []))
print("real answer, papers present:     ", grounding_precondition("some grounded answer", PAPERS))

assert grounding_precondition("INSUFFICIENT EVIDENCE: none found", PAPERS) is False
assert grounding_precondition("some grounded answer", []) is False
assert grounding_precondition("some grounded answer", PAPERS) is True

## Step 9 — a judge client that degrades to deterministic-only, same pattern as `04-llm-chunk-scoring.ipynb`

Production calls a dedicated grounding judge client (Gemini by default, a
separate quota from the answering model). This notebook reuses the same
Groq/OpenAI selection `04-retrieve/04-llm-chunk-scoring.ipynb` already
uses, under a spend ceiling -- and if neither key is set, `check_grounding`
below must fall back to the deterministic signal alone rather than fail.

In [ ]:
import os
import json as jsonlib


def get_judge_client():
    if os.environ.get("GROQ_API_KEY"):
        from groq import Groq
        return "groq", "llama-3.1-8b-instant", Groq(api_key=os.environ["GROQ_API_KEY"])
    if os.environ.get("OPENAI_API_KEY"):
        from openai import OpenAI
        return "openai", "gpt-4o-mini", OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    raise RuntimeError("no GROQ_API_KEY or OPENAI_API_KEY set")


def parse_judge_json(text: str):
    raw = (text or "").strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1] if "```" in raw[3:] else raw.lstrip("`")
        if raw.startswith("json"):
            raw = raw[4:]
    start, end = raw.find("{"), raw.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    try:
        return jsonlib.loads(raw[start : end + 1])
    except jsonlib.JSONDecodeError:
        return None


print(parse_judge_json('here is my answer: {"grounded": true, "score": 0.9, "unsupported_claims": []} thanks'))
print(parse_judge_json("not json at all"))

assert parse_judge_json("not json at all") is None

## Step 10 — `check_grounding`: the full orchestration

Ported from `check_grounding`. Runs the deterministic pass always; calls
the judge only if a key is available, under `nbio.cost_meter` since this
is a real paid call; merges the two signals with one hard rule kept
verbatim from production -- **a deterministic fabrication caps the score
at 0.5 no matter how confident the judge is.** A judge can be talked into
optimism about prose; it cannot talk its way out of a quote that provably
isn't in the evidence.

In [ ]:
async def check_grounding(query: str, answer_text: str, papers: list[dict], meter=None) -> dict | None:
    if not grounding_precondition(answer_text, papers):
        return None

    precheck = deterministic_check(answer_text, papers)

    judge = None
    judge_provider = judge_model = ""
    try:
        judge_provider, judge_model, client = get_judge_client()
        precheck_summary = jsonlib.dumps({
            "fabricated_quotes": precheck["unsupported_quotes"],
            "orphan_pmc_urls": precheck["orphan_urls"],
        })
        prompt = (
            "You are checking whether an answer is supported by retrieved evidence.\n"
            f"Question: {query}\nAnswer: {answer_text}\nEvidence: {context_text(papers)[:4000]}\n"
            f"Deterministic pre-check findings: {precheck_summary}\n"
            'Respond with JSON only: {"grounded": <bool>, "score": <0-1>, "unsupported_claims": [<str>, ...]}'
        )
        resp = client.chat.completions.create(
            model=judge_model, messages=[{"role": "user", "content": prompt}],
            temperature=0.0, max_tokens=400,
        )
        if meter is not None:
            usage = getattr(resp, "usage", None)
            meter.record(judge_model, getattr(usage, "prompt_tokens", 0) if usage else 0,
                         getattr(usage, "completion_tokens", 0) if usage else 0)
        judge = parse_judge_json(resp.choices[0].message.content)
    except Exception as exc:
        print(f"judge unavailable ({type(exc).__name__}): {exc}")
        judge = None

    unsupported = list(precheck["unsupported_quotes"])
    if judge:
        for c in judge.get("unsupported_claims", []):
            if not is_bibliographic(c, papers) and c not in unsupported:
                unsupported.append(c)
        score = float(judge.get("score", 0.0))
        if precheck["unsupported_quotes"] or precheck["orphan_urls"]:
            score = min(score, 0.5)
        grounded = bool(judge.get("grounded")) and not precheck["unsupported_quotes"] and not precheck["orphan_urls"]
        method = "deterministic+llm"
    else:
        has_problem = bool(precheck["unsupported_quotes"] or precheck["orphan_urls"])
        score = 0.5 if has_problem else 1.0
        grounded = None if not has_problem else False
        method = "deterministic"

    unsupported = [c for c in unsupported if not is_bibliographic(c, papers)]
    return {
        "grounded": grounded, "score": round(score, 3), "unsupported_claims": unsupported[:5],
        "orphan_urls": precheck["orphan_urls"], "method": method,
        "judge_provider": judge_provider, "judge_model": judge_model,
    }


with nbio.cost_meter(budget_usd=0.50) as meter:
    result = await check_grounding("burns?", clean_answer, PAPERS, meter=meter)
    print("clean answer, no known key required for this to be correct either way:")
    nbio.show_json(result)
    print()
    print(meter.report())

assert result["method"] == "deterministic" or result["method"] == "deterministic+llm"
if result["method"] == "deterministic":
    assert result["grounded"] is None  # no problem found, judge unavailable -- not "definitely grounded"

## Step 11 — the fabrication-caps-score rule, proven directly

Even a maximally confident judge (`grounded: true, score: 0.95`) cannot
rescue an answer whose deterministic pass already found a fabricated
quote. This is the rule from Step 10 that matters most; here it's checked
against a synthetic judge response rather than a real call, so it's
provable with no key at all.

In [ ]:
def merge_with_fake_judge(precheck: dict, fake_judge: dict) -> float:
    score = float(fake_judge["score"])
    if precheck["unsupported_quotes"] or precheck["orphan_urls"]:
        score = min(score, 0.5)
    return score


precheck_with_fabrication = deterministic_check(fabricated_answer, PAPERS)
overconfident_judge = {"grounded": True, "score": 0.95, "unsupported_claims": []}

capped_score = merge_with_fake_judge(precheck_with_fabrication, overconfident_judge)
print(f"judge said score=0.95, grounded=True -- but the deterministic pass found a fabricated quote")
print(f"final score after the cap: {capped_score}")

assert capped_score <= 0.5

## Step 12 — `annotation_for`: the user-facing trailer

Ported verbatim. This is what a specialist's answer actually shows a
reader when grounding is weak -- everything above exists to decide
whether this ever gets appended.

In [ ]:
def annotation_for(result: dict | None, threshold: float = 0.7) -> str:
    if not result:
        return ""
    weak = (result.get("grounded") is False) or (result.get("score", 1.0) < threshold)
    if not weak:
        return ""
    bits = []
    if result.get("unsupported_claims"):
        bits.append("unsupported — " + "; ".join(result["unsupported_claims"]))
    if result.get("orphan_urls"):
        bits.append("citations not in retrieved set: " + ", ".join(result["orphan_urls"]))
    detail = (" " + " | ".join(bits)) if bits else ""
    return f"\n\n\u26A0\uFE0F GROUNDING ({result.get('score')}/1.0):{detail} Treat the above as low-confidence."


strong_result = {"grounded": True, "score": 0.9}
weak_result = {"grounded": False, "score": 0.3, "unsupported_claims": ["claim x"], "orphan_urls": []}

print("strong result annotation:", repr(annotation_for(strong_result)))
print("weak result annotation:  ", annotation_for(weak_result))
print("no result at all:        ", repr(annotation_for(None)))

assert annotation_for(strong_result) == ""
assert "GROUNDING" in annotation_for(weak_result) and "claim x" in annotation_for(weak_result)
assert annotation_for(None) == ""

## Where this runs in the pipeline

Between `04-retrieve` and `06-bench`: it takes a query, an answer, and the
retrieved context `04-retrieve` produced, and it is the reason
`06-bench`'s faithfulness measurements mean anything at all -- a
benchmark that measures faithfulness with no grounding check upstream is
measuring a property nothing in the pipeline enforces.

## What did not come across

`EvidenceTable.tsx` and `evidenceParser.ts` -- the product's UI rendering
of a grounding result -- are frontend, not machinery, and out of scope
here. The LLM judge's real prompt template (`prompt_manager.render(...,
"grounding_check", ...)`) is product-specific YAML; Step 9/10 use a
minimal inline prompt instead, in the same spirit as
`04-retrieve/04-llm-chunk-scoring.ipynb`'s own inline `RCS_PROMPT`.